# P20 · Optimización exploratoria de una cartera de estudios para carga bip!

> **¿Qué combinación de zonas conviene estudiar primero bajo un presupuesto y una regla de diversidad explícitos?**

Diez celdas candidatas de P16, derivadas de GTFS DTPM y lugares bip! públicos.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/optimizacion_carga_bip/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# 1. Detectar el entorno y obtener una copia limpia del repositorio.
# Esta celda también funciona localmente si se abre el notebook dentro del repo.
import os
import platform
import subprocess
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules
if not Path('pyproject.toml').exists():
    destino_repo = Path('lab-dtpm')
    if not destino_repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git', str(destino_repo)], check=True)
    os.chdir(destino_repo)

ROOT = Path.cwd().resolve()
PAQUETES = ['pandas>=2.2', 'numpy>=1.26', 'openpyxl', 'XlsxWriter', 'matplotlib', 'polars>=1.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PAQUETES], check=True)
print('Entorno:', 'Google Colab' if EN_COLAB else 'Python local/Jupyter')
print('Python:', platform.python_version())
print('Repositorio:', ROOT)


In [ ]:
# 2. Importar y registrar versiones: evita resultados imposibles de auditar.
import matplotlib
import numpy as np
import pandas as pd
import polars as pl

print({'pandas': pd.__version__, 'polars': pl.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__})


## Alternativas para integrar los datos

La vía reproducible es clonar GitHub y ejecutar `scripts/fetch_data.py`, como hace la siguiente celda. Si la red institucional lo impide, consulte la [guía portable](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/docs/ejecucion-portable.md): explica carga desde el equipo, montaje de Google Drive, descarga directa y validación SHA-256. No cambie la fuente por datos simulados sin declararlo.


In [ ]:
# 3. Obtener las fuentes originales declaradas por el caso.
# Si la fuente externa falla, el error se conserva: no se sustituyen datos
# reales por sintéticos. Los CSV versionados permiten ejecutar el laboratorio
# comparativo, pero no reemplazan la adquisición del caso completo.
fuentes = []
for fuente in fuentes:
    try:
        subprocess.run([sys.executable, 'scripts/fetch_data.py', fuente], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'No se pudo obtener {fuente}. Revise la URL, la red y las alternativas de carga descritas arriba.'
        ) from exc


## 1 · El problema

P16 produce una lista corta, pero una lista no resuelve qué estudiar con un presupuesto
limitado. P20 formula esa elección como una cartera auditable y mantiene separados el
óptimo matemático, la evidencia observada y la decisión pública que todavía falta.


## 2 · La solución que se va a aplicar

Se identifican candidatos, se asignan tres bandas relativas de latitud, se enumeran
todas las combinaciones de uno a cinco estudios y se maximiza el puntaje P16 sujeto a
diversidad mínima. La frontera y sus tres mejores alternativas se publican completas.


## Implementaciones comparables

La ruta principal conserva el caso completo en Pandas. El laboratorio comparable ejecuta Pandas y Polars en este runtime sobre el mismo CSV. R usa el mismo contrato y se valida en CI; el runtime Python estándar de Colab no ejecuta R de forma nativa.

#### Contrato común del ejercicio

- **Objetivo.** Comparar el mejor puntaje acumulado alcanzable para presupuestos de uno a cinco estudios.
- **Fuente que leen las tres implementaciones.**
  [`resultados/frontera_presupuesto.csv`](resultados/frontera_presupuesto.csv), CSV UTF-8 versionado y producido
  por la resolución completa del caso. Los datos originales, su publicador, fecha,
  licencia, URL y huella se documentan en [`case.json`](case.json) y, cuando existe,
  `procedencia.json`. Este CSV es un **resultado reproducible**, no se presenta como
  una nueva fuente oficial.
- **Alcance.** mejor cartera factible para cada presupuesto. Los tres scripts comparables no sustituyen la adquisición,
  limpieza ni controles completos de los pasos Pandas: aíslan el indicador para que
  una diferencia observada sea atribuible al motor.
- **Fórmula o regla.** max Σ prioridad_i·x_i, con Σx_i=k y al menos min(k,3) bandas; x_i∈{0,1}. El desempate usa los ID en orden ascendente.
- **Interpretación.** La curva muestra valor bajo el puntaje de cribado P16; no representa utilidad social, dinero, demanda ni una orden de construcción.
- **Error frecuente.** Sumar prioridades sin declarar diversidad concentra la cartera; llamar beneficio al puntaje inventa una unidad económica.

| Campo | Tipo observado | Función en el ejercicio |
| --- | --- | --- |
| `presupuesto` | `int64` | categoría/eje X |
| `zonas_minimas` | `int64` | campo auxiliar del contrato |
| `candidatos` | `str` | campo auxiliar del contrato |
| `puntaje_total` | `float64` | medida/eje Y |
| `zonas_representadas` | `int64` | campo auxiliar del contrato |
| `paradas_sin_cobertura_referenciales` | `int64` | campo auxiliar del contrato |

#### Qué cambia entre motores

| Dimensión | Pandas | Polars | R (`dplyr`/`readr`) |
| --- | --- | --- | --- |
| Modelo | DataFrame mutable por operaciones | expresiones columnares y plan optimizable | tubería declarativa sobre `tibble` |
| Nulos | `NaN`/`NA`; `groupby` exige decidir `dropna` | `null` separado de `NaN` | `NA`; muchas funciones exigen `na.rm` explícito |
| Orden | varias operaciones lo conservan, pero se ordena explícitamente | no se presupone tras agrupar | se fija con `arrange` |
| Gráfico | Matplotlib | Matplotlib tras extraer sólo X/Y | ggplot2 y su gramática de capas |
| Ventaja típica | ecosistema amplio y aprendizaje gradual | paralelismo, tipos y consultas grandes | estadística, comunicación y análisis reproducible |
| Límite aquí | el índice puede introducir alineaciones implícitas | añade una dependencia y otra semántica de tipos | requiere runtime y paquetes R independientes |

La equivalencia se valida sobre CSV normalizados con tolerancia numérica `1e-9`; no
se exige que la sintaxis ni los píxeles sean idénticos. Para ejecutar en un entorno
limpio consulte [`docs/ejecucion-portable.md`](../../docs/ejecucion-portable.md).

### Pandas + Matplotlib

```python
"""Laboratorio comparable Pandas + Matplotlib del caso optimizacion_carga_bip.

Lee el mismo contrato CSV que las versiones Polars y R. La resolución completa del
caso permanece en ``solution.py`` y en los pasos del cuaderno; este archivo aísla el
indicador central para que la comparación entre motores sea controlada.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Buscar la raíz desde el propio archivo evita rutas C:\... específicas del autor y
# funciona igual después de clonar el repositorio en Linux, Windows o Google Colab.
ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "optimizacion_carga_bip" / "resultados" / "frontera_presupuesto.csv"
REQUIRED = {"presupuesto", "puntaje_total"}


def analizar() -> pd.DataFrame:
    """Carga, valida y transforma el indicador; no modifica el archivo de entrada."""
    datos = pd.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # La expresión conserva explícitos filtro, agrupación, orden y selección para que
    # el estudiante pueda alinearlos con las expresiones de Polars y dplyr.
    resultado = datos.sort_values("presupuesto")[["presupuesto", "puntaje_total"]]
    return resultado.reset_index(drop=True)


def graficar(resultado: pd.DataFrame, destino: Path) -> Path:
    """Representa exactamente las columnas comparadas y explicita sus unidades."""
    x = resultado["presupuesto"].tolist()
    y = resultado["puntaje_total"].tolist()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, y, marker="o", linewidth=2, color="#11253B")
    ax.set_xlabel("Estudios financiables (recuento)")
    ax.set_ylabel("Puntaje exploratorio acumulado")
    ax.set_ylim(bottom=0)
    ax.set_title("Frontera exploratoria según presupuesto de estudios")
    ax.grid(axis="y", alpha=0.25)
    fig.text(0.01, 0.01, "Fuente: Resultado reproducible P16; no representa demanda ni beneficio monetario · Período: escenario pedagógico P20", fontsize=8, color="#58677A")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado.to_string(index=False))
    if args.salida:
        resultado.to_csv(args.salida, index=False)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `python cases/optimizacion_carga_bip/cuaderno/solucion_pandas.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/optimizacion_carga_bip/cuaderno/solucion_pandas.py', '--grafico', 'solucion_pandas-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_pandas-colab.png'))


### Polars + Matplotlib

```python
"""Laboratorio comparable Polars + Matplotlib del caso optimizacion_carga_bip.

Usa el mismo CSV, columnas, reglas y escala que Pandas y R. Polars expresa las
transformaciones como un plan de expresiones columnares; esa diferencia de método es
el objeto de aprendizaje, no una razón para cambiar la pregunta.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "optimizacion_carga_bip" / "resultados" / "frontera_presupuesto.csv"
REQUIRED = {"presupuesto", "puntaje_total"}


def analizar() -> pl.DataFrame:
    """Carga y valida antes de transformar para producir un error explicativo."""
    datos = pl.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # select al final fija el contrato de salida y evita comparar columnas auxiliares.
    return datos.sort("presupuesto").select(["presupuesto", "puntaje_total"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    """Convierte sólo las series del gráfico; el análisis permanece en Polars."""
    x = resultado["presupuesto"].to_list()
    y = resultado["puntaje_total"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, y, marker="o", linewidth=2, color="#11253B")
    ax.set_xlabel("Estudios financiables (recuento)")
    ax.set_ylabel("Puntaje exploratorio acumulado")
    ax.set_ylim(bottom=0)
    ax.set_title("Frontera exploratoria según presupuesto de estudios")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Resultado reproducible P16; no representa demanda ni beneficio monetario · Período: escenario pedagógico P20",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.salida:
        resultado.write_csv(args.salida)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/optimizacion_carga_bip/cuaderno/solucion_polars.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/optimizacion_carga_bip/cuaderno/solucion_polars.py', '--grafico', 'solucion_polars-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_polars-colab.png'))


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Laboratorio comparable R + dplyr/readr + ggplot2 del caso optimizacion_carga_bip.
#
# La resolución completa sigue en solution.py y en los pasos Pandas. Este script
# cambia el motor sobre el mismo contrato CSV para estudiar sintaxis y semántica sin
# cambiar datos, indicador ni escala gráfica.

required <- c("readr", "dplyr")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  stop(
    "Faltan paquetes: ", paste(missing, collapse = ", "),
    ". Instálelos con install.packages(c('readr', 'dplyr', 'ggplot2'))."
  )
}

# Buscar pyproject.toml hacia arriba hace portable el script entre terminal, RStudio,
# Posit Cloud, IRkernel y un clon en Colab. LAB_ROOT permite una ruta explícita sin
# codificar la unidad o el directorio del desarrollador.
find_root <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "pyproject.toml"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("No se encontró la raíz del repositorio.")
    current <- parent
  }
}
root <- Sys.getenv("LAB_ROOT", unset = find_root())
source_csv <- file.path(root, "cases", "optimizacion_carga_bip", "resultados", "frontera_presupuesto.csv")

# readr conserva nombres y tipos previsibles. La validación temprana produce un error
# que explica el contrato roto, en vez de fallar más tarde dentro de una gráfica.
datos <- readr::read_csv(source_csv, show_col_types = FALSE, progress = FALSE)
required_columns <- c("presupuesto", "puntaje_total")
missing_columns <- setdiff(required_columns, names(datos))
if (length(missing_columns)) stop("Faltan columnas: ", paste(missing_columns, collapse = ", "))

# dplyr expresa la misma selección, filtro, agrupación y orden que las otras vías.
resultado <- datos |> dplyr::arrange(.data$presupuesto) |> dplyr::select(.data$presupuesto, .data$puntaje_total)
print(resultado)

args <- commandArgs(trailingOnly = TRUE)
value_after <- function(flag) {
  pos <- match(flag, args)
  if (is.na(pos)) return(NULL)
  if (pos == length(args)) stop(flag, " necesita una ruta de salida")
  args[pos + 1]
}

csv_output <- value_after("--salida")
if (!is.null(csv_output)) readr::write_csv(resultado, csv_output)

plot_output <- value_after("--grafico")
if (!is.null(plot_output)) {
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["presupuesto"]], y = .data[["puntaje_total"]])
  ) +
    ggplot2::geom_line(ggplot2::aes(group = 1), linewidth = 1, colour = "#11253B") +
    ggplot2::geom_point(size = 2.5, colour = "#55D6BE") +
    ggplot2::labs(
      title = "Frontera exploratoria según presupuesto de estudios",
      x = "Estudios financiables (recuento)", y = "Puntaje exploratorio acumulado",
      caption = "Fuente: Resultado reproducible P16; no representa demanda ni beneficio monetario · Período: escenario pedagógico P20"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(plot_output, grafico, width = 9, height = 5, dpi = 150)
  cat(plot_output, "\n")
}
```

Ejecutar con `Rscript cases/optimizacion_carga_bip/cuaderno/solucion.R`.


### Ejecutar R

Use `solucion.R` desde R/RStudio, Posit Cloud o Jupyter con IRkernel. También se incluye `solucion.Rmd`. En Colab, seleccione un runtime R compatible o use la instalación experimental documentada; se marca como compatibilidad documentada, no como ejecución validada.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.optimizacion_carga_bip import solution as portfolio

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Validar el contrato heredado de P16


La optimización sólo es trazable si conserva la fuente y el significado del puntaje.

> ⚠️ Tratar prioridad como demanda convierte una regla de cribado en una medición inexistente.


In [ ]:
datos = portfolio.cargar()
datos


### Paso 2 · Crear candidatos y bandas relativas


Un ID estable permite auditar selecciones; la banda introduce diversidad explícita.

> ⚠️ Llamar comuna a la banda inventaría una geografía administrativa.


In [ ]:
candidatos = portfolio.preparar_candidatos(datos)
candidatos[['candidato_id', 'celda_lat', 'celda_lon', 'banda_territorial', 'prioridad_exploratoria']]


### Paso 3 · Definir la variable de decisión


xᵢ vale 1 si la celda i entra en la cartera y 0 en caso contrario.

> ⚠️ Permitir valores fraccionarios describe medio estudio y cambia el problema.


In [ ]:
{'variable': 'x_i ∈ {0,1}', 'objetivo': 'max Σ prioridad_i × x_i', 'presupuesto': 'Σ x_i = k'}


### Paso 4 · Hacer visible la diversidad mínima


Para k=1,2,≥3 se exigen una, dos o tres bandas relativas.

> ⚠️ Agregar la restricción después de ver el ganador sería ajustar la regla al resultado.


In [ ]:
{k: min(k, 3) for k in range(1, 6)}


### Paso 5 · Enumerar las combinaciones factibles


Con diez candidatos, revisar exhaustivamente es simple y verificable.

> ⚠️ Usar una heurística sin necesidad puede ocultar una cartera mejor.


In [ ]:
frontera, alternativas = portfolio.optimizar(candidatos)
alternativas


### Paso 6 · Construir la frontera presupuestaria


Cada fila conserva el mejor objetivo bajo exactamente k estudios.

> ⚠️ Comparar carteras de distinto tamaño sin declarar presupuesto favorece siempre la más grande.


In [ ]:
frontera


### Paso 7 · Visualizar valor acumulado por presupuesto


La curva hace visible el incremento, no un beneficio monetario.

> ⚠️ Etiquetar el eje como rentabilidad atribuye dinero que el modelo no contiene.


In [ ]:
frontera[['presupuesto', 'puntaje_total']]


### Paso 8 · Separar estudio, factibilidad e inversión


El escenario de cuatro estudios es ilustrativo y requiere validación externa.

> ⚠️ Presentar coordenadas como sitios aprobados excede la evidencia y el alcance de P20.


In [ ]:
resultados = portfolio.analizar()
resultados['seleccion_presupuesto_4'][['candidato_id', 'banda_territorial', 'prioridad_exploratoria']]


## 4 · El resultado

El óptimo selecciona estudios bajo reglas declaradas; no selecciona predios ni autoriza inversión.

## 5 · Qué queda aprendido

1. Un modelo hereda los límites de su entrada.
2. Las restricciones expresan criterios.
3. La enumeración es auditable.
4. Óptimo no significa decisión final.
